In [1]:
"""
Cosmo-Norm Normalization Validation

This notebook validates that the cosmological normalization (cosmo_norm) 
produces properly normalized fields with mean≈0 and std≈1 after Z-score normalization.

The cosmo_norm pipeline:
1. Divide DM fields by Omega_m, baryonic fields by Omega_b
2. Apply log10(field + 1)
3. Apply Z-score normalization using computed stats

We expect the final normalized values to have mean≈0, std≈1.
"""
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent.parent))

# Paths
DATA_ROOT = '/mnt/home/mlee1/ceph/train_data_rotated2_128_cpu/train/'
STATS_DIR = '/mnt/home/mlee1/vdm_BIND/data/'

print("Loading cosmo_norm statistics...")

Loading cosmo_norm statistics...


In [2]:
# Load the computed statistics
stats = np.load(os.path.join(STATS_DIR, 'cosmo_norm_all_stats.npz'))

print("Cosmo-norm statistics:")
print(f"  DM input:     mean={stats['dm_input_mean']:.6f}, std={stats['dm_input_std']:.6f}")
print(f"  Large-scale:  mean={stats['large_scale_mean']:.6f}, std={stats['large_scale_std']:.6f}")
print(f"  DM target:    mean={stats['dm_target_mean']:.6f}, std={stats['dm_target_std']:.6f}")
print(f"  Gas:          mean={stats['gas_mean']:.6f}, std={stats['gas_std']:.6f}")
print(f"  Stellar:      mean={stats['star_mean']:.6f}, std={stats['star_std']:.6f}")

# Load quantile transformer for stellar
qt = joblib.load(os.path.join(STATS_DIR, 'cosmo_norm_quantile_normalizer_stellar.pkl'))
print(f"\nQuantile transformer: n_quantiles={len(qt.quantiles_)}")

Cosmo-norm statistics:
  DM input:     mean=10.424788, std=0.456662
  Large-scale:  mean=10.313859, std=0.397896
  DM target:    mean=10.357579, std=0.451831
  Gas:          mean=10.508808, std=0.391659
  Stellar:      mean=1.850961, std=3.501313

Quantile transformer: n_quantiles=1000


In [4]:
# Load file list from cache (much faster than scanning)
from vdm.astro_dataset import _get_cached_file_list

all_files = _get_cached_file_list(DATA_ROOT)
print(f"Found {len(all_files)} files")

# Random sample for validation
np.random.seed(123)
n_val = 1000
sample_indices = np.random.choice(len(all_files), size=n_val, replace=False)
sample_files = [all_files[i] for i in sample_indices]
print(f"Using {n_val} random samples for validation")

/mnt/sw/nix/store/gpkc8q6zjnp3n3h3w9hbmbj6gjbxs85w-python-3.10.10-view/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/mnt/sw/nix/store/6qvrglgqdpwhbw9zv2nh07fpd7a4wq31-py-torchvision-0.15.2/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


📂 Loading file list from cache: /mnt/home/mlee1/ceph/train_data_rotated2_128_cpu/train/file_list_cache.txt
   Loaded 408860 files in 0.48s
Found 408860 files
Using 1000 random samples for validation


In [ ]:
# Apply cosmo_norm pipeline to samples and collect normalized values
dm_input_norm = []
large_scale_norm = []
dm_target_norm = []
gas_norm = []
stellar_norm = []
stellar_quantile_norm = []

for fpath in sample_files:
    try:
        with np.load(fpath) as data:
            m_dm = data['condition']
            m_target = data['target']
            params = data['params']
            large_scale = data['large_scale']
            if large_scale.shape[0] == 4:
                large_scale = large_scale[1:]
            
            omega_m = params[0]
            omega_b = params[6]
            
            # Step 1: Cosmo normalization (divide by Omega_m/Omega_b)
            dm_cond_cosmo = m_dm / omega_m
            ls_cosmo = large_scale / omega_m
            dm_tgt_cosmo = m_target[0] / omega_m
            gas_cosmo = m_target[1] / omega_b
            star_cosmo = m_target[2] / omega_b
            
            # Step 2: Log transform
            dm_cond_log = np.log10(dm_cond_cosmo + 1)
            ls_log = np.log10(ls_cosmo + 1)
            dm_tgt_log = np.log10(dm_tgt_cosmo + 1)
            gas_log = np.log10(gas_cosmo + 1)
            star_log = np.log10(star_cosmo + 1)
            
            # Step 3: Z-score normalization
            dm_cond_z = (dm_cond_log - stats['dm_input_mean']) / stats['dm_input_std']
            ls_z = (ls_log - stats['large_scale_mean']) / stats['large_scale_std']
            dm_tgt_z = (dm_tgt_log - stats['dm_target_mean']) / stats['dm_target_std']
            gas_z = (gas_log - stats['gas_mean']) / stats['gas_std']
            star_z = (star_log - stats['star_mean']) / stats['star_std']
            
            # Also test quantile normalization for stellar
            star_log_noisy = star_log + np.random.randn(*star_log.shape) * 1e-4
            star_quantile = qt.transform(star_log_noisy.flatten().reshape(-1, 1)).flatten()
            
            # Collect samples (subsample to avoid memory issues)
            dm_input_norm.extend(dm_cond_z.flatten()[::4])
            large_scale_norm.extend(ls_z.flatten()[::4])
            dm_target_norm.extend(dm_tgt_z.flatten()[::4])
            gas_norm.extend(gas_z.flatten()[::4])
            stellar_norm.extend(star_z.flatten()[::4])
            stellar_quantile_norm.extend(star_quantile[::4])
            
    except Exception as e:
        continue

print(f"Collected {len(dm_input_norm)} samples per field")

# Convert to arrays
dm_input_norm = np.array(dm_input_norm)
large_scale_norm = np.array(large_scale_norm)
dm_target_norm = np.array(dm_target_norm)
gas_norm = np.array(gas_norm)
stellar_norm = np.array(stellar_norm)
stellar_quantile_norm = np.array(stellar_quantile_norm)

In [ ]:
# Print validation statistics
print("=" * 60)
print("VALIDATION: After cosmo_norm + log + Z-score normalization")
print("=" * 60)
print(f"{'Field':<20} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 60)

for name, arr in [
    ('DM input', dm_input_norm),
    ('Large-scale', large_scale_norm),
    ('DM target', dm_target_norm),
    ('Gas', gas_norm),
    ('Stellar (Z-score)', stellar_norm),
    ('Stellar (Quantile)', stellar_quantile_norm),
]:
    print(f"{name:<20} {arr.mean():>10.4f} {arr.std():>10.4f} {arr.min():>10.2f} {arr.max():>10.2f}")

print("\n✓ Expected: mean≈0, std≈1 for all fields")
print("✓ Quantile-normalized stellar should follow N(0,1)")

In [ ]:
# Plot distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Standard normal for reference
x_ref = np.linspace(-5, 5, 100)
y_ref = np.exp(-x_ref**2 / 2) / np.sqrt(2 * np.pi)

fields = [
    ('DM Input', dm_input_norm),
    ('Large-scale', large_scale_norm),
    ('DM Target', dm_target_norm),
    ('Gas', gas_norm),
    ('Stellar (Z-score)', stellar_norm),
    ('Stellar (Quantile)', stellar_quantile_norm),
]

for ax, (name, arr) in zip(axes.flat, fields):
    ax.hist(arr, bins=100, density=True, alpha=0.7, label=f'{name}')
    ax.plot(x_ref, y_ref, 'r--', lw=2, label='N(0,1)')
    ax.set_xlabel('Normalized value')
    ax.set_ylabel('Density')
    ax.set_title(f'{name}\nmean={arr.mean():.3f}, std={arr.std():.3f}')
    ax.legend()
    ax.set_xlim(-5, 5)

plt.tight_layout()
plt.savefig('cosmo_norm_validation.png', dpi=150)
plt.show()

print("✓ Saved: cosmo_norm_validation.png")

In [ ]:
# Compare with original (non-cosmo) normalization stats
print("\n" + "=" * 60)
print("COMPARISON: Cosmo-norm vs Original normalization stats")
print("=" * 60)

# Load original stats
orig_dm = np.load(os.path.join(STATS_DIR, 'dark_matter_normalization_stats.npz'))
orig_gas = np.load(os.path.join(STATS_DIR, 'gas_normalization_stats.npz'))
orig_star = np.load(os.path.join(STATS_DIR, 'stellar_normalization_stats.npz'))

print(f"\n{'Field':<15} {'Original Mean':>15} {'Cosmo Mean':>15} {'Ratio':>10}")
print("-" * 55)

# Note: original stats are for log10(field + 1), cosmo for log10(field/param + 1)
print(f"{'DM target':<15} {float(orig_dm['dm_mag_mean']):>15.4f} {float(stats['dm_target_mean']):>15.4f} {float(orig_dm['dm_mag_mean'])/float(stats['dm_target_mean']):>10.3f}")
print(f"{'Gas':<15} {float(orig_gas['gas_mag_mean']):>15.4f} {float(stats['gas_mean']):>15.4f} {float(orig_gas['gas_mag_mean'])/float(stats['gas_mean']):>10.3f}")
print(f"{'Stellar':<15} {float(orig_star['star_mag_mean']):>15.4f} {float(stats['star_mean']):>15.4f} {float(orig_star['star_mag_mean'])/float(stats['star_mean']):>10.3f}")

print("\nNote: Dividing by Omega_m (~0.3) or Omega_b (~0.05) before log transform")
print("      increases the values, so mean should be higher for cosmo_norm.")

In [ ]:
# Visualize a single sample before and after normalization
sample_file = sample_files[0]
with np.load(sample_file) as data:
    m_dm = data['condition']
    m_target = data['target']
    params = data['params']
    
    omega_m = params[0]
    omega_b = params[6]
    
print(f"Sample file: {os.path.basename(sample_file)}")
print(f"Omega_m = {omega_m:.4f}, Omega_b = {omega_b:.5f}")

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

# Row 1: DM
axes[0, 0].imshow(m_dm, cmap='magma')
axes[0, 0].set_title('DM (raw)')
axes[0, 1].imshow(m_dm / omega_m, cmap='magma')
axes[0, 1].set_title(f'DM / Ω_m ({omega_m:.3f})')
axes[0, 2].imshow(np.log10(m_dm / omega_m + 1), cmap='magma')
axes[0, 2].set_title('log10(DM/Ω_m + 1)')
dm_norm = (np.log10(m_dm / omega_m + 1) - stats['dm_input_mean']) / stats['dm_input_std']
axes[0, 3].imshow(dm_norm, cmap='RdBu_r', vmin=-3, vmax=3)
axes[0, 3].set_title('Z-score normalized')

# Row 2: Gas
axes[1, 0].imshow(m_target[1], cmap='magma')
axes[1, 0].set_title('Gas (raw)')
axes[1, 1].imshow(m_target[1] / omega_b, cmap='magma')
axes[1, 1].set_title(f'Gas / Ω_b ({omega_b:.4f})')
axes[1, 2].imshow(np.log10(m_target[1] / omega_b + 1), cmap='magma')
axes[1, 2].set_title('log10(Gas/Ω_b + 1)')
gas_norm = (np.log10(m_target[1] / omega_b + 1) - stats['gas_mean']) / stats['gas_std']
axes[1, 3].imshow(gas_norm, cmap='RdBu_r', vmin=-3, vmax=3)
axes[1, 3].set_title('Z-score normalized')

# Row 3: Stars
axes[2, 0].imshow(m_target[2], cmap='magma')
axes[2, 0].set_title('Stars (raw)')
axes[2, 1].imshow(m_target[2] / omega_b, cmap='magma')
axes[2, 1].set_title(f'Stars / Ω_b ({omega_b:.4f})')
axes[2, 2].imshow(np.log10(m_target[2] / omega_b + 1), cmap='magma')
axes[2, 2].set_title('log10(Stars/Ω_b + 1)')
star_log = np.log10(m_target[2] / omega_b + 1)
star_log_noisy = star_log + np.random.randn(*star_log.shape) * 1e-4
star_qt = qt.transform(star_log_noisy.flatten().reshape(-1, 1)).reshape(star_log.shape)
axes[2, 3].imshow(star_qt, cmap='RdBu_r', vmin=-3, vmax=3)
axes[2, 3].set_title('Quantile normalized')

for ax in axes.flat:
    ax.axis('off')

plt.tight_layout()
plt.savefig('cosmo_norm_sample_visualization.png', dpi=150)
plt.show()

print("✓ Saved: cosmo_norm_sample_visualization.png")